# config_loader\nGenerated from 00_common/config_loader.py for Databricks notebook import.\n

In [ ]:
"""Control metadata loaders for CSV or Databricks table-backed configuration."""\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport os\nimport re\nimport threading\nimport warnings\nfrom pathlib import Path\nfrom typing import Any\n\n# Supports both ${VAR} and ${VAR:default} — consistent with global_config.py.\n_ENV_PATTERN = re.compile(r"\$\{([A-Z0-9_]+)(?::([^}]*))?\}")\n\n# Module-level cache: absolute CSV path → resolved rows.\n# Avoids re-reading the same file on every per-entity metadata call.\n# _CSV_LOCK serialises concurrent reads/writes so the cache is safe when\n# multiple threads call load_csv_config() at the same time.\n_CSV_CACHE: dict[str, list[dict[str, str]]] = {}\n_CSV_LOCK = threading.Lock()\n\n\ndef clear_config_cache() -> None:\n    """Clear the in-memory CSV row cache.\n\n    Useful in tests (call in setUp/teardown) or when you need hot-reload\n    of config files without restarting the Python process.\n    """\n    with _CSV_LOCK:\n        _CSV_CACHE.clear()\n\n\ndef _substitute_env(value: str) -> str:\n    def repl(match: re.Match[str]) -> str:\n        key = match.group(1)\n        default_value = match.group(2) if match.group(2) is not None else ""\n        return os.getenv(key, default_value)\n\n    return _ENV_PATTERN.sub(repl, value)\n\n\ndef _resolve_row_env(row: dict[str, str]) -> dict[str, str]:\n    return {k: _substitute_env(v) if isinstance(v, str) else v for k, v in row.items()}\n\n\ndef load_csv_config(file_path: str) -> list[dict[str, str]]:\n    abs_path = str(Path(file_path).resolve())\n    with _CSV_LOCK:\n        if abs_path in _CSV_CACHE:\n            return _CSV_CACHE[abs_path]\n    try:\n        with open(abs_path, "r", encoding="utf-8") as f:\n            rows = [_resolve_row_env(row) for row in csv.DictReader(f)]\n    except FileNotFoundError:\n        raise FileNotFoundError(\n            f"Framework control CSV not found: {abs_path}. "\n            "Ensure the file exists in your config directory."\n        ) from None\n    with _CSV_LOCK:\n        _CSV_CACHE[abs_path] = rows\n    return rows\n\ndef get_lifecycle_log_table(config: dict) -> str:\n    return config.get("lifecycle_log_table", "audit.entity_lifecycle_log")\n\n\ndef _to_row_dicts_from_table(spark, table_name: str) -> list[dict[str, str]]:\n    rows = spark.table(table_name).collect()\n    out: list[dict[str, str]] = []\n    for row in rows:\n        out.append({k: "" if v is None else str(v) for k, v in row.asDict().items()})\n    return out\n\n\ndef _metadata_mode(global_config: dict[str, Any] | None) -> str:\n    return str((global_config or {}).get("metadata", {}).get("mode", "csv")).lower()\n\n\ndef _metadata_csv_root(config_dir: str, global_config: dict[str, Any] | None) -> Path:\n    csv_root = (global_config or {}).get("metadata", {}).get("csv_root", "")\n    if csv_root:\n        return Path(config_dir).resolve().parent / str(csv_root)\n    return Path(config_dir)\n\n\ndef _table_name_for(global_config: dict[str, Any], key: str) -> str:\n    return str(global_config.get("metadata", {}).get("tables", {}).get(key, ""))\n\n\ndef load_control_rows(\n    config_dir: str,\n    control_name: str,\n    spark=None,\n    global_config: dict[str, Any] | None = None,\n) -> list[dict[str, str]]:\n    mode = _metadata_mode(global_config)\n\n    if mode == "table":\n        if spark is None:\n            raise ValueError("Metadata mode is table but spark session is not provided")\n        table_name = _table_name_for(global_config or {}, control_name)\n        if not table_name:\n            raise ValueError(f"Missing table name config for {control_name}")\n        return _to_row_dicts_from_table(spark, table_name)\n\n    csv_root = _metadata_csv_root(config_dir, global_config)\n    csv_path = csv_root / f"{control_name}.csv"\n    return load_csv_config(str(csv_path))\n\n\ndef _active_flag(row: dict[str, str]) -> bool:\n    return row.get("is_active", "").strip().lower() == "true"\n\n\ndef _matches_filters(row: dict[str, str], filters: dict[str, str | None]) -> bool:\n    for key, value in filters.items():\n        if value and row.get(key, "").strip().lower() != value.strip().lower():\n            return False\n    return True\n\n\ndef active_sources(\n    config_dir: str,\n    spark=None,\n    global_config: dict[str, Any] | None = None,\n    product_name: str | None = None,\n    source_system: str | None = None,\n    source_entity: str | None = None,\n) -> list[dict[str, str]]:\n    rows = load_control_rows(config_dir, "source_registry", spark=spark, global_config=global_config)\n    return [\n        r\n        for r in rows\n        if _active_flag(r)\n        and _matches_filters(\n            r,\n            {\n                "product_name": product_name,\n                "source_system": source_system,\n                "source_entity": source_entity,\n            },\n        )\n    ]\n\n\ndef mappings_for_entity(\n    config_dir: str,\n    source_system: str,\n    source_entity: str,\n    product_name: str | None = None,\n    spark=None,\n    global_config: dict[str, Any] | None = None,\n) -> list[dict[str, str]]:\n    rows = load_control_rows(config_dir, "column_mapping", spark=spark, global_config=global_config)\n    return [\n        r\n        for r in rows\n        if _active_flag(r)\n        and r.get("source_system", "").lower() == source_system.lower()\n        and r.get("source_entity", "").lower() == source_entity.lower()\n        and (not product_name or r.get("product_name", "").lower() == product_name.lower())\n    ]\n\n\ndef dq_rules_for_entity(\n    config_dir: str,\n    source_system: str,\n    source_entity: str,\n    product_name: str | None = None,\n    spark=None,\n    global_config: dict[str, Any] | None = None,\n) -> list[dict[str, str]]:\n    rows = load_control_rows(config_dir, "dq_rules", spark=spark, global_config=global_config)\n    return [\n        r\n        for r in rows\n        if _active_flag(r)\n        and r.get("source_system", "").lower() == source_system.lower()\n        and r.get("source_entity", "").lower() == source_entity.lower()\n        and (not product_name or r.get("product_name", "").lower() == product_name.lower())\n    ]\n\n\ndef publish_rule_for_entity(\n    config_dir: str,\n    source_system: str,\n    source_entity: str,\n    product_name: str | None = None,\n    spark=None,\n    global_config: dict[str, Any] | None = None,\n) -> dict[str, str] | None:\n    rows = load_control_rows(config_dir, "publish_rules", spark=spark, global_config=global_config)\n    for row in rows:\n        if (\n            _active_flag(row)\n            and\n            row.get("source_system", "").lower() == source_system.lower()\n            and row.get("source_entity", "").lower() == source_entity.lower()\n            and (not product_name or row.get("product_name", "").lower() == product_name.lower())\n        ):\n            return row\n    return None\n\n\ndef parse_json_cell(value: str | None) -> dict[str, Any]:\n    if not value or not value.strip():\n        return {}\n    try:\n        parsed = json.loads(value)\n        if isinstance(parsed, dict):\n            return parsed\n        warnings.warn(f"parse_json_cell: expected a JSON object, got {type(parsed).__name__}. Returning {{}}.")\n    except json.JSONDecodeError as exc:\n        warnings.warn(f"parse_json_cell: invalid JSON ({exc}). Value: {value!r:.120}. Returning {{}}.")\n    return {}\n\n\ndef parse_json_list(value: str | None) -> list[Any]:\n    if not value or not value.strip():\n        return []\n    try:\n        parsed = json.loads(value)\n        if isinstance(parsed, list):\n            return parsed\n        warnings.warn(f"parse_json_list: expected a JSON array, got {type(parsed).__name__}. Returning [].")\n    except json.JSONDecodeError as exc:\n        warnings.warn(f"parse_json_list: invalid JSON ({exc}). Value: {value!r:.120}. Returning [].")\n    return []\n